# Spin dynamics with states that stay in the engine

This notebook chains a DMRG ground state, a local operator, time evolution and correlation
functions with `quantx_engine.mps`, and never passes a state through Python: each MPS stays in
the native engine, under a handle, until it is freed. It computes the spin correlation function

$$C_j(t) = \langle \mathrm{GS} |\, S^z_j(t)\, S^z_c(0) \,| \mathrm{GS} \rangle$$

of the XXZ chain. It checks it against an exact diagonalisation at 10 sites, then draws its light
cone at 32 sites and the dynamical structure factor $S(k, \omega)$ of the Heisenberg chain, whose
weight sits inside the two-spinon continuum.

It needs the native engine, which the `pip install quantx-engine` wheels carry on Linux, macOS and
Windows, and runs in about two minutes on two CPU threads.

In [ ]:
import itertools
import time

import matplotlib.pyplot as plt
import numpy as np

import quantx_engine
from quantx_engine import mps
from quantx_engine.models import XXZChain

assert quantx_engine.backend_info()["native"], "this notebook needs the native engine"

## 1. Ten sites, against exact diagonalisation

`ground_state` returns an `MPS`. `copy().apply("Sz", c)` makes $S^z_c|\mathrm{GS}\rangle$, a
second state; `evolve` advances both by two-site TDVP, and `matrix_elements` gives
$\langle \mathrm{GS}(t) | S^z_j | \phi(t) \rangle$ at every site, which is $C_j(t)$: the ground
state's phase $e^{-iE_0 t}$ is carried by the bra.

The exact reference diagonalises $H = J \sum_i (S^x_i S^x_{i+1} + S^y_i S^y_{i+1} + \Delta S^z_i
S^z_{i+1})$ in the $S^z = 0$ sector, 252 states, and evolves $S^z_c |\mathrm{GS}\rangle$ in its
eigenbasis.

In [ ]:
L, c, delta, dt, n_steps = 10, 4, 0.5, 0.05, 60

def xxz_sector(L, delta):
    # H on the Sz = 0 product states (0 = up, 1 = down, site 0 first), and each state's S^z_j.
    basis = [s for s in itertools.product((0, 1), repeat=L) if sum(1 - 2 * b for b in s) == 0]
    index = {s: n for n, s in enumerate(basis)}
    H = np.zeros((len(basis),) * 2)
    for n, s in enumerate(basis):
        for i in range(L - 1):
            H[n, n] += delta * (0.5 - s[i]) * (0.5 - s[i + 1])
            if s[i] != s[i + 1]:
                t = list(s)
                t[i], t[i + 1] = s[i + 1], s[i]
                H[index[tuple(t)], n] += 0.5
    return H, np.array([[0.5 - b for b in s] for s in basis])

H, sz = xxz_sector(L, delta)
E, V = np.linalg.eigh(H)
g = V[:, 0]
times = dt * np.arange(n_steps + 1)
w = V.T @ (sz[:, c] * g)                                    # S^z_c |GS> in the eigenbasis
C_ed = np.array([np.exp(1j * E[0] * t) * (g * (V @ (np.exp(-1j * E * t) * w))) @ sz
                 for t in times])

chain = XXZChain(L=L, delta=delta)
gs = mps.ground_state(chain, max_bond=64, cutoff=1e-14, n_sweeps=40, tol=1e-13)
phi = gs.copy().apply("Sz", c)
C = [gs.matrix_elements("Sz", phi)]
for _ in range(n_steps):
    gs.evolve(dt=dt, max_bond=64, cutoff=1e-14)
    phi.evolve(dt=dt, max_bond=64, cutoff=1e-14)
    C.append(gs.matrix_elements("Sz", phi))
C = np.array(C)
err = np.abs(C - C_ed).max()
print(f"E0: MPS {gs.dmrg.energy:.12f}, exact {E[0]:.12f}")
print(f"largest |C - C_exact| over {n_steps} steps and {L} sites: {err:.1e}")
assert abs(gs.dmrg.energy - E[0]) < 1e-10 and err < 1e-6

## 2. Thirty-two sites: the light cone

Past exact diagonalisation, the same calls run at 32 sites. A spin flip at the centre spreads
ballistically: the change $C_j(t) - C_j(0)$ is confined to a cone whose edge moves at the
spin-wave velocity, $v = \pi J/2$ at the Heisenberg point. (The static part $C_j(0)$ decays only
as a power law, the chain being critical, so it fills the whole chain from the start.)

In [ ]:
L, c, dt, n_steps, chi = 32, 16, 0.05, 120, 64
chain = XXZChain(L=L, delta=1.0)
t0 = time.time()
gs = mps.ground_state(chain, max_bond=chi)
phi = gs.copy().apply("Sz", c)
C = [gs.matrix_elements("Sz", phi)]
for _ in range(n_steps):
    gs.evolve(dt=dt, max_bond=chi)
    phi.evolve(dt=dt, max_bond=chi)
    C.append(gs.matrix_elements("Sz", phi))
C = np.array(C)
times = dt * np.arange(n_steps + 1)
print(f"{n_steps} steps at chi = {chi} in {time.time() - t0:.0f} s; bonds now up to {max(phi.bonds)}")

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(np.abs(C - C[0]), origin="lower", aspect="auto", cmap="magma",
               extent=(-0.5, L - 0.5, 0, times[-1]))
v = np.pi / 2
ax.plot([c - v * times[-1], c], [times[-1], 0], "w--", lw=1)
ax.plot([c + v * times[-1], c], [times[-1], 0], "w--", lw=1, label=r"$|j - c| = \pi t / 2$")
ax.set_xlim(-0.5, L - 0.5)
ax.set_xlabel("site j")
ax.set_ylabel("time t")
ax.legend(loc="upper right")
fig.colorbar(im, label=r"$|C_j(t) - C_j(0)|$")
plt.show()

# The cone: four sites ahead of its edge, the change is a few percent of its size inside.
D = np.abs(C - C[0])
dist = np.abs(np.arange(L)[None, :] - c)
ahead, inside = dist > v * times[:, None] + 4, dist <= v * times[:, None]
print(f"largest change inside the cone {D[inside].max():.3f}, four sites ahead of it "
      f"{D[ahead].max():.3f}")
assert D[ahead].max() < 0.05 * D[inside].max()

## 3. The structure factor and the two-spinon continuum

A Fourier transform in space and time gives $S(k, \omega)$ (a Gaussian window in time sets the
resolution, and $C_j(-t) = C_j(t)^*$ supplies negative times). For the Heisenberg chain its weight
lies in the two-spinon continuum, between the des Cloizeaux–Pearson dispersion $\omega_L(k) =
\frac{\pi J}{2} |\sin k|$ and $\omega_U(k) = \pi J |\sin(k/2)|$, and it peaks at the lower edge,
where in the infinite chain it diverges.

In [ ]:
T = times[-1]
win = np.exp(-0.5 * (times / (0.4 * T)) ** 2)               # time window
Ct = np.concatenate([np.conj(C[:0:-1]), C]) * np.concatenate([win[:0:-1], win])[:, None]
tt = np.concatenate([-times[:0:-1], times])
ks = 2 * np.pi * np.arange(1, L // 2) / L
ws = np.linspace(0, 4, 241)
r = np.arange(L) - c
phase_k = np.exp(-1j * np.outer(ks, r))                     # (k, j)
phase_w = np.exp(1j * np.outer(ws, tt))                     # (w, t)
S = np.real(phase_w @ Ct @ phase_k.T).T * dt                # S[k, w]

fig, ax = plt.subplots(figsize=(6, 4))
ax.pcolormesh(ks, ws, np.clip(S, 0, None).T, cmap="viridis", shading="auto")
ax.plot(ks, np.pi / 2 * np.abs(np.sin(ks)), "w-", lw=1, label=r"$\omega_L = \frac{\pi}{2}|\sin k|$")
ax.plot(ks, np.pi * np.abs(np.sin(ks / 2)), "w--", lw=1, label=r"$\omega_U = \pi|\sin(k/2)|$")
ax.set_xlabel("k")
ax.set_ylabel(r"$\omega$")
ax.legend(loc="upper left")
plt.show()

# At each k, the largest weight lies inside the continuum, and at its lower edge, to the time
# window's resolution of about 1 / (0.4 T).
res = 1 / (0.4 * T)
peak = ws[np.argmax(S, axis=1)]
lo, hi = np.pi / 2 * np.abs(np.sin(ks)), np.pi * np.abs(np.sin(ks / 2))
print("k      peak   continuum")
for k, p, a, b in zip(ks, peak, lo, hi):
    print(f"{k:.3f}  {p:.3f}  [{a:.3f}, {b:.3f}]")
assert np.all((peak > lo - res) & (peak < hi + res)) and np.all(np.abs(peak - lo) < res)

Every state above lived in the engine: the five calls of each step moved only their requests and
the measured numbers. The states are freed when their Python objects are collected, or with
`free()`.